# 08 — The CARLA campaign

The same question as notebook 05, asked of a real road network:

> Does the recommended policy beat a fixed route, and is the difference real?

Notebook 05 answered it on a generated graph where a leg takes exactly as long
as the model predicted. Here the bus drives, traffic gets in the way, and a
leg takes whatever it takes. **This is the transfer test** — the one that says
whether a month of graph experiments described anything real.

**This notebook needs a running CARLA server, and it is slow.** A trial runs
close to real time, so a full campaign is hours, not minutes. It resumes from
the results store, so it can be run in sittings.

```bash
docker compose -f compose_software.yml up -d
```

In [1]:
# --- Bootstrap ------------------------------------------------------------
# This notebook imports the project as `src.…`, which requires the repository
# root on the import path. `./jupyter-run.sh` puts it there; opening the file
# any other way does not. This cell finds the root by walking up from the
# working directory, so the notebook runs wherever it is launched from and
# whatever folder it sits in.
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Seeded runs only reproduce when Python's string hashing is pinned: several
# policies break ties with min() over a set, whose iteration order is
# randomised per process otherwise. The simulation refuses to run unpinned;
# this says so now rather than after a long cell.
if os.environ.get("PYTHONHASHSEED") in (None, "random"):
    print("WARNING: PYTHONHASHSEED is not pinned, so seeded runs will be "
          "refused.\n         Launch with ./jupyter-run.sh, or set "
          "PYTHONHASHSEED=0.")

print(f"repository root: {ROOT}")

repository root: /home/danny/Desktop/האוניברסיטה הפתוחה/סדנה/autobus


In [2]:
import os

# How much work this notebook does. Every notebook in this folder has this
# switch and honours the AUTOBUS_SCALE environment variable, so nothing here
# needs editing to run it bigger or smaller.
#
#   "quick"  the default: minutes, same conclusions, wider error bars
#   "full"   the sample sizes behind the figures in the submitted documents
#   "smoke"  a tiny end-to-end check that the notebook runs at all
#
# The outputs saved in this file were produced at "full".
SCALE = os.environ.get("AUTOBUS_SCALE", "quick")
print(f"running at SCALE = {SCALE!r}")

running at SCALE = 'quick'


In [3]:
import matplotlib.pyplot as plt
from cycler import cycler

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

In [4]:
import socket

CARLA_HOST = os.environ.get("CARLA_HOST", "localhost")
CARLA_PORT = int(os.environ.get("CARLA_PORT", "2000"))


def server_is_up(host=CARLA_HOST, port=CARLA_PORT, timeout=2.0):
    """
    Report whether something is listening on the CARLA port.
    :param host: CARLA host.
    :param port: CARLA RPC port.
    :param timeout: Seconds to wait for the connection.
    :return: True when the port accepts a connection.
    """
    try:
        with socket.create_connection((host, port), timeout):
            return True
    except OSError:
        return False


SERVER = server_is_up()
print(f"CARLA at {CARLA_HOST}:{CARLA_PORT} -- "
      + ("up" if SERVER else "NOT REACHABLE"))
if not SERVER:
    print("\nCells that need the server will skip themselves and say so.\n"
          "Start it with:  docker compose -f compose_software.yml up -d")

CARLA at localhost:2000 -- up


## 1. What is prepared, and what is missing

No server needed for this: it reads the files in the repository.

A town is ready when it has a topology dump, an authored stop list, and a
`traffic_efficiency` that is not 1.0 — the default meaning "never calibrated",
which would have the policy plan against a bus that achieves the posted speed
limit exactly.

In [5]:
import pandas as pd

from src.map.CarlaMapProvider import CarlaMapProvider
from src.map.carla_stops import stops_path
from src.map.carla_topology import topology_path

ALL_TOWNS = ["Town01", "Town02", "Town03", "Town05", "TAU"]


def readiness(town):
    """
    Report whether one town can be campaigned on.
    :param town: Town name.
    :return: A row describing what is present.
    """
    has_topology = topology_path(town).exists()
    has_stops = stops_path(town).exists()
    row = {"town": town, "topology": has_topology, "stops": has_stops,
           "traffic_efficiency": None, "stop count": None, "ready": False}
    if has_topology and has_stops:
        provider = CarlaMapProvider(town)
        row["traffic_efficiency"] = round(provider.traffic_efficiency(), 4)
        row["stop count"] = len(provider.get_stops())
        row["ready"] = provider.traffic_efficiency() != 1.0
    return row


readiness_table = pd.DataFrame([readiness(t) for t in ALL_TOWNS]).set_index("town")
display(readiness_table)

READY_TOWNS = [t for t in ALL_TOWNS if readiness(t)["ready"]]
print(f"ready: {READY_TOWNS}")

,topology,stops,traffic_efficiency,stop count,ready
town,,,,,
Town01,True,True,0.4751,15,True
Town02,True,True,0.4645,15,True
Town03,True,True,0.5980,15,True
Town05,True,True,0.3309,15,True
TAU,True,True,0.7032,25,True


ready: ['Town01', 'Town02', 'Town03', 'Town05', 'TAU']


### If a town is not ready

Two preparation steps, both needing a server, both one-off:

```bash
# 1. dump the roads (writes src/map/carla_maps/<town>.topology.json)
./run.sh -m src.preparation.carla_dump --town Town01

# 2. fit traffic_efficiency by driving three tours of every stop pair
./run.sh -m src.preparation.carla_calibrate --town Town01 --tours 3
```

Authoring the stop list is the one step nothing can do for you — it is a
decision about where a bus should stop, and it lives in
`<town>.stops.json`.

## 2. The campaign

Three policies, because CARLA time is expensive and the interesting comparison
is narrow: the **baseline**, the **insertion policy**, and **insertion plus
reordering**. The service window is 750 s and passenger patience is also
750 s — the recommended configuration, in which no accepted request is ever
abandoned. Section 6 covers what that makes measurable and what it does not.

Everything about the world that could change a result goes into the stored run
identity, including three tags that are not parameters of the simulation at
all:

- **`traffic_efficiency`** — a pre-calibration run must never answer for a
  post-calibration one.
- **`steering`** — how the bus was driven. Without it, a run recorded under
  the old turn-list steering shares an identity with one driven by CARLA's
  agent, and the campaign would skip the new trial as "already recorded".
- **`routing`** — what the planner optimised and whether it could change lane.

Each of those was added after a real incident of two incomparable runs pooling
silently. **This is the part of the harness that took the most learning.**

In [6]:
# The first two prepared towns at "quick". Note that the second of them,
# Town02, cannot discriminate -- the fixed route already serves every request
# there in every seed -- so a run meant to compare policies is better off
# swapping it for TAU:
#
#     TOWNS = ["Town01", "TAU"] if SCALE == "quick" else READY_TOWNS
#
# See the closing section for what that costs and what it buys.
TOWNS = {"smoke": READY_TOWNS[:1], "quick": READY_TOWNS[:2],
         "full": READY_TOWNS}[SCALE]
SEEDS = {"smoke": range(1), "quick": range(3), "full": range(10)}[SCALE]

# The scenario, matching the campaign these figures come from. The window is
# set EQUAL to passenger patience, which is the recommended configuration and
# has a consequence for what can be measured -- see section 6.
WINDOW = 750.0
PATIENCE = 750.0
MAX_REQUESTS = {"smoke": 4, "quick": 15, "full": 30}[SCALE]

TRIAL = dict(max_requests=MAX_REQUESTS, demand_interval_s=90.0, capacity=20,
             max_wait_s=PATIENCE, dwell_s=10.0, dt=1.0,
             # Generous on purpose: a ceiling that truncates a run turns its
             # wait time into a fact about the ceiling rather than the policy.
             max_time=40_000.0)

NPC_VEHICLES = 20
FIXED_DELTA_S = 0.05

print(f"{len(TOWNS)} towns x {len(list(SEEDS))} seeds x 3 policies "
      f"= {len(TOWNS) * len(list(SEEDS)) * 3} trials")
print("expect roughly 5-10 minutes per trial with a server up")

2 towns x 3 seeds x 3 policies = 18 trials
expect roughly 5-10 minutes per trial with a server up


In [7]:
import os
from pathlib import Path

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment.store import DEFAULT_RUNS_DIR, load_runs, to_results
from src.simulation.carla_world import GENERATED_SPEED_CEILING_KPH, is_generated

RUNS = Path(os.environ.get("AUTOBUS_RUNS_DIR", DEFAULT_RUNS_DIR))

POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Window W750", lambda g: WindowInsertionPolicy(g, max_wait_s=WINDOW)),
    ("Reordering W750",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW)),
]


def speed_limit_tag(town):
    """
    How this town's speed limits were derived, for the run identity.

    A shipped town publishes speed-limit landmarks; a generated one does not,
    so its limits come from the OpenDRIVE file and are capped. Two different
    procedures, and pooling them would be exactly the trap the other tags
    exist to prevent.
    :param town: Town name.
    :return: The identity tag.
    """
    if is_generated(town):
        return f"xodr-capped-{int(GENERATED_SPEED_CEILING_KPH)}-v1"
    return "landmark-harmonic-v1"


def world_for(town, provider):
    """
    Build the stored identity for one town's world.
    :param town: Town name.
    :param provider: That town's loaded map provider.
    :return: The world identity.
    """
    return {
        "town": town, "backend": "CarlaBackend",
        "npc_vehicles": NPC_VEHICLES, "fixed_delta_s": FIXED_DELTA_S,
        "no_rendering": True, "blueprint": "vehicle.volkswagen.t2_2021",
        "traffic_efficiency": provider.traffic_efficiency(),
        "speed_limits": speed_limit_tag(town),
        "steering": "direct-drive-v2",
        "routing": "time-lane-change-v1",
    }

In [8]:
RUN_CAMPAIGN = True     # set False to analyse what is already recorded

if not SERVER:
    print("skipped: needs a live CARLA server")
elif not RUN_CAMPAIGN:
    print("RUN_CAMPAIGN = False -- using whatever is already in the store")
else:
    import time

    from tqdm.auto import tqdm

    from src.experiment import run_campaign
    from src.simulation.CarlaBackend import (
        CarlaBackend, speed_limits_from, stop_positions_from)

    def backend_factory_for(town, positions, speed_for):
        """
        Build a per-seed world builder for one town.
        :param town: Town to drive in.
        :param positions: That town's stop coordinates.
        :param speed_for: That town's speed-limit lookup.
        :return: A function of seed returning a backend factory.
        """
        def for_seed(seed):
            def build(graph, start_stop_id):
                """Build the CARLA backend for one trial."""
                return CarlaBackend(
                    stop_positions=positions, position=start_stop_id,
                    town=town, speed_for=speed_for, lane_changes=True,
                    host=CARLA_HOST, port=CARLA_PORT,
                    fixed_delta_s=FIXED_DELTA_S, dwell_s=TRIAL["dwell_s"],
                    npc_vehicles=NPC_VEHICLES, seed=seed, no_rendering=True,
                    # CARLA results are not independent of server history: an
                    # earlier campaign lost seven runs to a
                    # back-to-back-trials artefact and had to be discarded.
                    reload_world=True)
            return build
        return for_seed

    failures = []
    for town in tqdm(TOWNS, desc="towns", unit="town"):
        provider = CarlaMapProvider(town)
        positions = stop_positions_from(provider)
        world = world_for(town, provider)
        for_seed = backend_factory_for(
            town, positions, speed_limits_from(provider))

        trials = tqdm(total=len(list(SEEDS)) * len(POLICIES), unit="trial",
                      desc=f"  {town}", leave=False)
        # Seed-major, so the three policies of a seed finish together: an
        # interrupted campaign then leaves paired data rather than an orphan.
        for seed in SEEDS:
            for label, factory in POLICIES:
                started = time.perf_counter()

                # A CARLA trial is minutes of real driving. Without a bar
                # inside it, the one above moves once per trial and a stalled
                # server is indistinguishable from a slow one. The observer
                # rides through run_campaign into run_trial; it is excluded
                # from a run's identity, so watching cannot change what gets
                # recorded, and it works here only because a CARLA campaign
                # is necessarily n_jobs=1 -- a callback cannot cross into a
                # worker process.
                leg = tqdm(total=TRIAL["max_requests"], unit="req",
                           desc=f"    seed {seed} {label}", leave=False)
                seen = {"settled": 0}

                def observe(progress, bar=leg, seen=seen):
                    """Advance the trial's bar as passengers finish."""
                    if progress.settled == seen["settled"]:
                        return
                    seen["settled"] = progress.settled
                    bar.n = progress.settled
                    bar.set_postfix_str(
                        f"{progress.sim_time / 60:.0f} min simulated, "
                        f"{progress.pending} waiting", refresh=False)
                    bar.refresh()

                try:
                    outcome = run_campaign(
                        policies=[(label, factory)], seeds=[seed],
                        world=world, directory=RUNS, provider=provider,
                        backend_factory_for=for_seed,
                        on_tick=observe,
                        # One server cannot host a campaign per core, and
                        # run_campaign refuses parallelism for this world.
                        n_jobs=1, progress=False, **TRIAL)
                    if not outcome.ran:
                        # Nothing ticked, because nothing ran.
                        leg.set_postfix_str("already recorded", refresh=False)
                except Exception as error:          # noqa: BLE001
                    # Broad on purpose: a CARLA failure surfaces as almost
                    # anything, and losing the remaining trials is worse.
                    failures.append((town, seed, label, repr(error)))
                    tqdm.write(f"  {town} seed {seed} {label}: {error!r}")
                finally:
                    leg.close()
                trials.update(1)
                trials.set_postfix_str(
                    f"seed {seed} {label} "
                    f"({(time.perf_counter() - started) / 60:.1f} min)")
        trials.close()

    print(f"\n{len(failures)} failures")

towns:   0%|          | 0/2 [00:00<?, ?town/s]

  Town01:   0%|          | 0/9 [00:00<?, ?trial/s]

    seed 0 Fixed route:   0%|          | 0/15 [00:00<?, ?req/s]

    seed 0 Window W750:   0%|          | 0/15 [00:00<?, ?req/s]

    seed 0 Reordering W750:   0%|          | 0/15 [00:00<?, ?req/s]

    seed 1 Fixed route:   0%|          | 0/15 [00:00<?, ?req/s]

... (12 outputs left out)

    seed 2 Fixed route:   0%|          | 0/15 [00:00<?, ?req/s]

    seed 2 Window W750:   0%|          | 0/15 [00:00<?, ?req/s]

    seed 2 Reordering W750:   0%|          | 0/15 [00:00<?, ?req/s]


0 failures


## 3. Collect

Rebuilt from the store rather than from memory, so hours of driving can be
tabulated again without repeating any of it.

In [9]:
from src.experiment.summary import summarize_results

from IPython.display import Markdown


def collect(town):
    """
    Rebuild one town's runs from the store.
    :param town: Town name.
    :return: Policy label to trial results, paired by seed.
    """
    provider = CarlaMapProvider(town)
    return to_results(load_runs(RUNS, where={
        "identity.world.town": town,
        "identity.world.backend": "CarlaBackend",
        "identity.world.steering": "direct-drive-v2",
        "identity.world.traffic_efficiency": provider.traffic_efficiency(),
        "identity.trial.max_requests": TRIAL["max_requests"],
        "identity.trial.capacity": TRIAL["capacity"],
    }))


collected = {town: collect(town) for town in TOWNS}
for town, results in collected.items():
    counts = {label: len(runs) for label, runs in results.items()}
    print(f"{town}: {counts}")

Town01: {'Fixed route': 3, 'Reordering W750': 3, 'Window W750': 3}
Town02: {'Fixed route': 3, 'Reordering W750': 3, 'Window W750': 3}


In [10]:
if not any(collected.values()):
    print("nothing collected yet -- run section 2 with a server up")

for town, results in collected.items():
    if not results:
        continue
    display(Markdown(f"### {town}"))
    display(Markdown(summarize_results(results).markdown))

### Town01

| Property | Fixed route | Reordering W750 | Window W750 |
| --- | --- | --- | --- |
| Fulfilled requests | 9.7 (±39.3%) | 14.0 (±17.7%) | 14.3 (±20.0%) |
| Cancelled requests | 5.3 (±71.1%) | 1.0 (±248.4%) | 0.7 (±430.3%) |
| Mean wait time | 262.8 s (±41.7%) | 317.7 s (±34.6%) | 334.0 s (±2.2%) |
| P90 wait time | 544.6 s (±34.5%) | 592.2 s (±34.3%) | 590.8 s (±30.0%) |
| Mean ride time | 561.8 s (±66.3%) | 241.6 s (±75.0%) | 277.5 s (±90.9%) |
| Fulfillment rate | 64.4 % (±39.3%) | 93.3 % (±17.7%) | 95.6 % (±20.0%) |
| Served fraction | 64.4 % (±39.3%) | 93.3 % (±17.7%) | 95.6 % (±20.0%) |
| Throughput | 0.3 req/min (±57.8%) | 0.4 req/min (±43.2%) | 0.4 req/min (±26.3%) |
| Tick snapshots | 2327.0 (±38.2%) | 1919.3 (±27.1%) | 1952.7 (±14.5%) |
| Fulfilled (wait samples) | 9.7 (±39.3%) | 14.0 (±17.7%) | 14.3 (±20.0%) |
| Min wait time | 24.6 s (±111.1%) | 57.0 s (±146.3%) | 52.3 s (±204.6%) |
| Max wait time | 635.3 s (±17.6%) | 622.3 s (±38.1%) | 712.3 s (±8.2%) |
| Min ride time | 57.4 s (±167.6%) | 46.5 s (±116.4%) | 38.3 s (±50.6%) |
| Max ride time | 1075.9 s (±5.0%) | 801.5 s (±188.4%) | 671.0 s (±53.3%) |

### Town02

| Property | Fixed route | Reordering W750 | Window W750 |
| --- | --- | --- | --- |
| Fulfilled requests | 15.0 (±0.0%) | 14.7 (±9.8%) | 15.0 (±0.0%) |
| Cancelled requests | 0.0 (±0.0%) | 0.3 (±430.3%) | 0.0 (±0.0%) |
| Mean wait time | 375.2 s (±34.5%) | 219.8 s (±20.4%) | 228.2 s (±16.0%) |
| P90 wait time | 610.8 s (±21.0%) | 355.8 s (±47.1%) | 388.4 s (±46.1%) |
| Mean ride time | 341.4 s (±8.7%) | 152.4 s (±37.5%) | 144.9 s (±38.5%) |
| Fulfillment rate | 100.0 % (±0.0%) | 97.8 % (±9.8%) | 100.0 % (±0.0%) |
| Served fraction | 100.0 % (±0.0%) | 97.8 % (±9.8%) | 100.0 % (±0.0%) |
| Throughput | 0.4 req/min (±7.8%) | 0.5 req/min (±15.3%) | 0.5 req/min (±20.2%) |
| Tick snapshots | 2125.7 (±7.9%) | 1763.3 (±13.5%) | 1775.3 (±19.5%) |
| Fulfilled (wait samples) | 15.0 (±0.0%) | 14.7 (±9.8%) | 15.0 (±0.0%) |
| Min wait time | 31.5 s (±125.8%) | 16.6 s (±351.1%) | 15.4 s (±391.7%) |
| Max wait time | 653.3 s (±22.5%) | 573.4 s (±64.1%) | 547.6 s (±22.7%) |
| Min ride time | 39.5 s (±189.3%) | 23.7 s (±37.0%) | 23.9 s (±40.1%) |
| Max ride time | 638.2 s (±16.9%) | 412.9 s (±15.2%) | 389.6 s (±20.9%) |

## 4. Does the headline result transfer?

The design review's criterion again: beat the fixed route on at least one KPI,
no worse on the others, at least 80% served.

In [11]:
import numpy as np


def percentage_differences(results, baseline, contender):
    """
    Percentage difference per KPI, signed so that positive always means better.
    :param results: Policy label to trial results.
    :param baseline: The fixed-route comparator.
    :param contender: The policy under test.
    :return: KPI name to signed percentage difference.
    """
    def mean(label, get):
        return float(np.mean([get(t.kpi) for t in results[label]]))

    differences = {}
    for name, get in [("wait", lambda k: k.mean_wait_time),
                      ("ride", lambda k: k.mean_ride_time)]:
        base, arm = mean(baseline, get), mean(contender, get)
        differences[name] = 100 * (base - arm) / base
    base, arm = (mean(baseline, lambda k: k.throughput),
                 mean(contender, lambda k: k.throughput))
    differences["throughput"] = 100 * (arm - base) / base
    return differences


def verdict(results, baseline="Fixed route", contender="Reordering W750"):
    """
    Check the design review's success criteria (§3.2.2) for one town.

    Criterion 2 is the SUM of the percentage differences across wait, ride and
    throughput -- not a test per KPI, so ground given on one measure may be
    bought back on another.
    :param results: Policy label to trial results.
    :param baseline: The policy that must be beaten.
    :param contender: The policy under test.
    :return: A summary row, or None when the town has no paired runs.
    """
    if baseline not in results or contender not in results:
        return None

    def mean(label, get):
        return float(np.mean([get(t.kpi) for t in results[label]]))

    fulfillment = mean(contender, lambda k: k.fulfillment_rate)
    served = mean(contender, lambda k: k.served_fraction)
    differences = percentage_differences(results, baseline, contender)
    total = sum(differences.values())

    row = {
        "n": len(results[contender]),
        "fulfillment": f"{fulfillment:.1%}",
        "1. >= 80%": "PASS" if fulfillment >= 0.80 else "FAIL",
    }
    row.update({f"{name} %": f"{value:+.1f}"
                for name, value in differences.items()})
    row["sum %"] = f"{total:+.1f}"
    row["2. sum > 0"] = "PASS" if total > 0 else "FAIL"
    row["served"] = f"{served:.1%}"
    row["vs fixed"] = (
        f"{100 * (served - mean(baseline, lambda k: k.served_fraction)):+.1f} pp")
    return row


rows = {town: verdict(results) for town, results in collected.items()}
rows = {town: row for town, row in rows.items() if row is not None}
if rows:
    display(pd.DataFrame(rows).T)
else:
    print("no paired runs in the store yet -- run section 2 with a server up")

,n,fulfillment,1. >= 80%,wait %,ride %,throughput %,sum %,2. sum > 0,served,vs fixed
Town01,3,93.3%,PASS,-20.9,+57.0,+74.6,+110.7,PASS,93.3%,+28.9 pp
Town02,3,97.8%,PASS,+41.4,+55.4,+18.0,+114.8,PASS,97.8%,-2.2 pp


In [12]:
from scipy import stats

paired_rows = []
for town, results in collected.items():
    if "Window W750" not in results or "Reordering W750" not in results:
        continue
    l0 = np.array([t.kpi.mean_ride_time for t in results["Window W750"]])
    l1 = np.array([t.kpi.mean_ride_time for t in results["Reordering W750"]])
    n = min(len(l0), len(l1))
    if n < 2:
        paired_rows.append({"town": town, "n": n, "note": "too few runs"})
        continue
    test = stats.ttest_rel(l1[:n], l0[:n])
    paired_rows.append({
        "town": town, "n": n,
        "L1 - L0 ride s": round(float((l1[:n] - l0[:n]).mean()), 1),
        "p": f"{test.pvalue:.3f}",
    })

if paired_rows:
    display(pd.DataFrame(paired_rows).set_index("town"))
else:
    print("no paired L0/L1 runs in the store yet")

,n,L1 - L0 ride s,p
town,,,
Town01,3,-35.9,0.499
Town02,3,7.5,0.024


**Expect this comparison to be under-powered**, and say so
rather than reporting a p-value as if it settled something. At ten seeds a
difference of tens of seconds in ride time is not resolvable — notebook 05
needed a thousand paired runs to establish it on the graph. The CARLA campaign
is there to show that the *headline* result transfers, not to re-derive the
fine structure.

## 5. Was the bus healthy?

The KPIs above are meaningless if the bus spent the run stuck. Every CARLA run
records what the vehicle actually did, and a campaign that does not read those
counters is not measuring routing — it is measuring how often the simulation
went wrong.

In [13]:
health = []
for town, results in collected.items():
    for label, runs in results.items():
        for run in runs:
            stats_row = {"town": town, "policy": label}
            stats_row.update({k: v for k, v in run.backend_stats.items()})
            health.append(stats_row)

if health:
    health_frame = pd.DataFrame(health)
    numeric = health_frame.select_dtypes("number").columns
    display(health_frame.groupby("town")[list(numeric)].mean().round(2))
else:
    print("no runs collected yet")

,arrivals,distance_m,falls_recovered,legs_started,obstructions_removed,queued_ticks,red_light_ticks,repaths,route_drops,stops_without_shoulder,ticks,unroutable_legs,unwedges,walker_spawn_failures,walkers_removed,walkers_spawned
town,,,,,,,,,,,,,,,,
Town01,24.22,7824.27,0.0,24.56,0.0,337.67,12252.22,0.0,0.0,0.0,41330.00,0.0,0.0,0.0,15.0,15.0
Town02,31.44,5098.42,0.0,31.89,1.0,748.67,12739.44,0.0,0.0,0.0,37682.22,0.0,0.0,0.0,15.0,15.0


## 6. Was the promise kept?

The insertion policies promise a pickup inside the service window. On the graph
that promise is kept by construction; in CARLA it could be broken, because the
drive can take longer than the estimate the schedule was certified against.

**But this scenario cannot measure that, and the reason is worth stating.** The
window is set equal to passenger patience — the recommended configuration —
so a breach is impossible *by construction*: anyone who would have waited
longer than the window has already given up and been cancelled. Abandonment
enforces the promise on the policy's behalf.

So this section reports a **check, not a rate**. A non-zero count here would
mean something is wrong with the promise itself, not with the traffic. To
measure a real breach rate the window has to be set *below* patience, which is
what notebook 04 does on the graph, where thousands of trials are affordable.

In [14]:
breach_rows = []
for town, results in collected.items():
    for label, runs in results.items():
        if "Fixed" in label:
            continue
        waits = [w for run in runs for w in run.wait_times]
        if not waits:
            continue
        breach_rows.append({
            "town": town, "policy": label, "fulfilled": len(waits),
            "worst wait s": round(max(waits)),
            "window s": WINDOW,
            "over the window": int(sum(w > WINDOW for w in waits)),
        })

if breach_rows:
    frame = pd.DataFrame(breach_rows).set_index(["town", "policy"])
    display(frame)
    total_over = frame["over the window"].sum()
    print("expected: 0 -- window == patience, so a late pickup cancels "
          "instead of being served")
    print(f"measured: {total_over}"
          + ("" if total_over == 0 else "   <- investigate, this should not happen"))
else:
    print("no fulfilled requests recorded yet")

fulfilled  worst wait s  window s  over the window
town   policy                                                             
Town01 Reordering W750         42           728     750.0                0
       Window W750             43           728     750.0                0
Town02 Reordering W750         44           681     750.0                0
       Window W750             45           589     750.0                0

expected: 0 -- window == patience, so a late pickup cancels instead of being served
measured: 0


## 7. Where this campaign sits on the occupancy axis

Notebook 06 found that demand-responsive routing stops paying above roughly
60% mean seat occupancy. That is a scope condition on **every** result in this
project, including this one, so it has to be checked here rather than assumed.

Occupancy is estimated per policy — not for the busiest arm, which would
report a number belonging to a policy that is not under test.

In [15]:
occupancy_rows = []
for town, results in collected.items():
    for label, runs in results.items():
        values = [
            (run.kpi.throughput / 60) * run.kpi.mean_ride_time
            / TRIAL["capacity"]
            for run in runs
            if run.kpi.throughput and run.kpi.mean_ride_time
        ]
        if values:
            occupancy_rows.append({"town": town, "policy": label,
                                   "occupancy %": round(100 * np.mean(values), 1)})

if not occupancy_rows:
    print("no runs to measure occupancy from yet")
else:
    occupancy_frame = pd.DataFrame(occupancy_rows).pivot(
        index="town", columns="policy", values="occupancy %")
    display(occupancy_frame)
    peak = occupancy_frame.max().max()
    print(f"peak {peak:.0f}% -- "
          + ("comfortably inside the safe regime"
             if peak < 60 else "past the ~60% line; read notebook 06 first"))

policy,Fixed route,Reordering W750,Window W750
town,,,
Town01,11.4,8.6,10.1
Town02,12.1,6.3,6.1


peak 12% -- comfortably inside the safe regime


## What this campaign establishes

Read it against the tables above rather than from memory: what a `"quick"` run
shows and what the full campaign showed are different, and both belong here.

### From this run

- **The driving was healthy.** No falls, no re-paths, no dropped routes, no
  unroutable legs across every trial — so the KPIs describe routing rather
  than a simulation going wrong.
- **The promise held.** Zero fulfilled passengers waited longer than the
  window, which is what "window == patience" guarantees by construction.
- **Occupancy is far inside the safe regime** (see §7), so notebook 06's
  scope condition does not apply to these numbers.
- **The fine structure does not resolve at three seeds.** The L1-versus-L0
  ride comparison is noise here — one town even shows a "significant"
  p-value on a 7.5 s difference, which is what n=3 does. Notebook 05 needed a
  thousand paired runs on the graph to establish that effect.

### A caution about the towns in a quick run

`"quick"` takes the first two prepared towns, and the second of them is
**Town02, where the fixed route already serves every request in every seed.**
Nothing can beat a perfect score, so that town's coverage comparison measures
nothing — the recommended policy comes out 2.2 points *behind*, and the number
is an artefact of a scenario too easy to discriminate rather than a finding.

It does illustrate one thing well. The design review's criterion is a **sum**
across the other three KPIs, not a test on each, and Town02 is exactly the case
it was written for: routing gives up 2.2 points of coverage and buys 41% of
wait, 55% of ride and 18% of throughput. Both criteria pass.

**For a run meant to discriminate, put TAU in the town list** — see below.

### From the full campaign, at ten seeds across five towns

These are prior results, not this notebook's output, and are quoted as such:

- **The headline result transfers.** The recommended policy beats the fixed
  route on coverage by margins far larger than the graph experiments produced
  — real road networks punish a fixed cycle harder than a generated graph.
- **TAU passes with the largest margin of any town**, and needed no recoveries.
  It is the strongest evidence that the pipeline is not tuned to CARLA's
  shipped towns, since nobody tuned CARLA for a Tel Aviv campus.

### Not tested here

Notebook 06's occupancy scope condition. This scenario runs at ~12% seat
occupancy, nowhere near the line; the question is asked properly by the
sustained-load campaigns, which drop the bus to 12 seats and raise demand
until it saturates. Their answer — that the inversion does **not** reproduce
on real geometry, in either patience regime — is a separate result and is
reported with the caveats it deserves.

Next: **09 — the harness**, which is how any of these numbers can be trusted
twice.